# Check report by hand

In [ ]:
import subprocess
import sys
from pathlib import Path

import Scriptum
import yaml

# Derived from where this notebook sits, so moving the worktree does not
# turn the check below into a false alarm. Jupyter starts in the notebook's
# own directory, and this cell runs before anything changes it.
NOTEBOOK_DIR = Path.cwd()
WORKTREE = NOTEBOOK_DIR.parents[3]
here = Path(Scriptum.__file__).resolve().parent.parent

print('python    ', sys.executable)
print('Scriptum  ', Path(Scriptum.__file__).resolve())
print('PyYAML    ', yaml.__version__)
print('branch    ', subprocess.run(['git', 'branch', '--show-current'], cwd=here,
                                   capture_output=True, text=True).stdout.strip())

if here != WORKTREE:
    print()
    print('WARNING: Scriptum is NOT coming from this worktree.')
    print('         Expected', WORKTREE)
    print("         Pick this worktree's .venv kernel - see tests/Instructions.md.")
else:
    print()
    print('OK: importing from this worktree.')


## A workspace

The run writes a deck and needs its data beside it, so everything is copied
to a temp directory. The repo stays clean, and you can throw the directory away.


In [ ]:
import os
import shutil
import tempfile

REPORT_DIR = WORKTREE / 'tests' / '02_basetest' / 'pptx-basic' / 'simple'
DATA_SOURCE = WORKTREE / 'tests' / 'data_source'

def workspace():
    """A fresh directory holding the fixtures, the template and the data."""
    work = Path(tempfile.mkdtemp(prefix='scriptum-'))
    for pattern in ('*.yaml', 'template.pptx'):
        for path in REPORT_DIR.glob(pattern):
            shutil.copy(path, work)
    shutil.copytree(DATA_SOURCE, work / 'data', dirs_exist_ok=True)
    os.chdir(work)
    return work

WORK = workspace()
print('working in', WORK)
print(sorted(p.name for p in WORK.iterdir()))


## 1. Read the document

`ReportDataFile` reads a `.yaml` document through `Scriptum.rdf.loader`;
anything else is refused with a message. The `.rdf` text parser is gone.

A broken document raises `DocumentError` carrying **every** diagnostic, not
just the first.


In [ ]:
rdf = Scriptum.ReportDataFile('powerpoint_simple.yaml')

print('documenttype:', rdf.settings.documenttype)
print('datadir     :', rdf.settings.datadir)
print('tasks       :', len(rdf.tasks))
print('errors      :', rdf.errors or 'none')


## 2. What the tasks say

`what` is the operation -- for a deck every top-level entry is a `copy`: the
layout of that name is cloned into a new slide. `where` is the marker an *add*
lands at, `target` the placeholder or tag name in the layout, and the address
the **instance**: `:titlecontent::2` is the second slide made from the
`TitleContent` layout.

Note the `_global_` tasks at the end: global fills are applied last, and the
task list carries that rule so no back end has to remember it.


In [ ]:
def show_tasks(tasks, limit=None, only=None):
    rows = [t for t in tasks if only is None or only in '.'.join(t.myAddress)]
    print(f'{"#":>4}  {"what":6} {"where":18} {"target":22} address')
    print('-' * 110)
    for t in rows[:limit]:
        print(f'{t.serial:>4}  {t.what or "-":6} {t.where or "-":18} '
              f'{t.target or "-":22} {".".join(t.myAddress)}')
    if limit and len(rows) > limit:
        print(f'... {len(rows) - limit} more')

show_tasks(rdf.tasks, limit=40)


Try `show_tasks(rdf.tasks, only=':titlecontent')` to see the two slides made
from one layout -- the first carries the table, the second the two pudding
pictures.


In [ ]:
show_tasks(rdf.tasks, only=':titlecontent')


## 3. Build the deck

The steps mirror `common_case.run_pptx_case`, which every pptx case test
uses: `artist` fills the layouts, `remove_slide(0)` drops the template's own
first slide. `finish` and `createpdf` stay off, as in the automated tests.

Anything the back end could not place prints a `WARNING`. A clean run prints
none.


In [ ]:
import contextlib
import io as _io

with contextlib.redirect_stdout(_io.StringIO()) as printed:
    managed = Scriptum.ManagedPptx('template.pptx')
    managed.artist(rdf, directfill=True, globalfill=True,
                   cleardust=True, setproperties=True)
    managed.remove_slide(0)
    managed.save('report.pptx', finish=False, createpdf=False)

complaints = [line for line in printed.getvalue().splitlines()
              if 'WARNING' in line or 'ERROR' in line]
print('written:', WORK / 'report.pptx')
print('complaints:', len(complaints))
for line in complaints[:20]:
    print('  ', line)


## 4. Read it back

Open `report.pptx` in PowerPoint if you want to look at it; this shows what it
*says* -- slide by slide, every text of every shape, then the cells of every
table -- which is what the automated comparison uses.


In [ ]:
import pptx

def spoken(path):
    said = []
    for slide in pptx.Presentation(path).slides:
        for shape in slide.shapes:
            if shape.has_text_frame:
                for paragraph in shape.text_frame.paragraphs:
                    said.append(''.join(run.text for run in paragraph.runs).strip())
            if shape.has_table:
                for row in shape.table.rows:
                    said.extend(cell.text.strip() for cell in row.cells)
    return [line for line in said if line]

lines = spoken(WORK / 'report.pptx')
print(len(lines), 'non-empty lines')
for line in lines[:30]:
    print('  ', line[:100])


## 5. Compare with the reference

`expected/powerpoint_simple.json`, beside this notebook, is what this
fixture's **`.rdf`** produced before the back end changed (the `.rdf` and its
parser are gone; the reference is their record). Digits and weekday names
are collapsed on both sides, because the reference was captured on another
day and `date: now` is evaluated per run.

`IDENTICAL` below means the YAML document says exactly what the text one said.


In [ ]:
import json
import re

DIGITS = re.compile(r'\d+')
WEEKDAY = re.compile(r'\b(?:Mon|Tue|Wed|Thu|Fri|Sat|Sun)\b')

def normalise(lines):
    return [WEEKDAY.sub('#', DIGITS.sub('#', line)) for line in lines]

REFERENCE = REPORT_DIR / 'expected' / 'powerpoint_simple.json'

expected = normalise(json.loads(REFERENCE.read_text(encoding='utf-8')))
got = normalise(spoken(WORK / 'report.pptx'))

print(f'reference {len(expected)} lines, this run {len(got)} lines')
if expected == got:
    print('IDENTICAL')
else:
    for i, (a, b) in enumerate(zip(expected, got)):
        if a != b:
            print(f'first difference at line {i}')
            print('  reference:', a[:110])
            print('  this run :', b[:110])
            break
    else:
        print('one is a prefix of the other')


## 6. What it shows

A text comparison cannot see a picture or a table. So: per slide, its layout
and the shapes that are not text. The sizes are what python-pptx computed
from each file and its tag; a picture placed at native size instead of the
tag's shows up here.


In [ ]:
from pptx.enum.shapes import MSO_SHAPE_TYPE
from pptx.util import Cm

def cm(length):
    return round(length / Cm(1), 2)

deck = pptx.Presentation(WORK / 'report.pptx')
for number, slide in enumerate(deck.slides, 1):
    print(f'slide {number}: {slide.slide_layout.name}')
    for shape in slide.shapes:
        kind = shape.shape_type
        if kind == MSO_SHAPE_TYPE.PICTURE:
            print(f'    picture  {cm(shape.width)} x {cm(shape.height)} cm  ({shape.image.content_type})')
        elif kind == MSO_SHAPE_TYPE.TABLE:
            print(f'    table    {len(shape.table.rows)} rows x {len(shape.table.columns)} columns')
        elif kind == MSO_SHAPE_TYPE.TEXT_BOX:
            print(f'    text box {shape.text_frame.text.strip()[:60]!r}')


## 7. The verdict

The same expectations `test_pptx_simple.py` holds, so a manual run ends with
a yes or no: one picture each on the title and definition slides, the 4 x 5
table standing alone on the first content slide, the two pudding pictures at
their sizes on the second, the text file with its extra fills on the material
slide, nothing on the back cover -- and the document's own title in the
deck's properties, which the runner used to overwrite with 'AutoReport'.


In [ ]:
slides = list(deck.slides)
layouts = [slide.slide_layout.name for slide in slides]
assert layouts == ['TitleSlide', 'TaskProjectDefinition', 'TitleContent',
                   'TitleContent', 'Material', 'BackCover'], layouts
title, definition, table_slide, pictures_slide, material, back = slides

def of_kind(slide, kind):
    return [shape for shape in slide.shapes if shape.shape_type == kind]

def sizes(slide):
    return [(cm(p.width), cm(p.height)) for p in of_kind(slide, MSO_SHAPE_TYPE.PICTURE)]

assert sizes(title) == [(7.99, 7.99)], sizes(title)
assert sizes(definition) == [(3.4, 3.4)], sizes(definition)

(table,) = of_kind(table_slide, MSO_SHAPE_TYPE.TABLE)
assert (len(table.table.rows), len(table.table.columns)) == (4, 5)
assert table.table.cell(0, 0).text == 'Type'

assert sizes(pictures_slide) == [(1.21, 1.24), (2.96, 3.04)], sizes(pictures_slide)

boxes = [shape.text_frame.text for shape in of_kind(material, MSO_SHAPE_TYPE.TEXT_BOX)]
assert boxes[0].startswith('A general text may look like this Lorem ipsum')

assert len(back.shapes) == 0, 'BackCover takes nothing from the document'
assert deck.core_properties.title == 'This is a bloody test document'

print('all checks passed -- the deck shows what it should' if expected == got
      else 'shapes are right, but the text differs from the reference (see above)')
